<a href="https://colab.research.google.com/github/myyyxa/ffmpy3/blob/master/RealESR_Upscale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-ESRGAN Inference Demo

[![arXiv](https://img.shields.io/badge/arXiv-Paper-<COLOR>.svg)](https://arxiv.org/abs/2107.10833)
[![GitHub Stars](https://img.shields.io/github/stars/xinntao/Real-ESRGAN?style=social)](https://github.com/xinntao/Real-ESRGAN)
[![download](https://img.shields.io/github/downloads/xinntao/Real-ESRGAN/total.svg)](https://github.com/xinntao/Real-ESRGAN/releases)

This is a **Practical Image Restoration Demo** of our paper [''Real-ESRGAN: Training Real-World Blind Super-Resolution with Pure Synthetic Data''](https://arxiv.org/abs/2107.10833).
We extend the powerful ESRGAN to a practical restoration application (namely, Real-ESRGAN), which is trained with pure synthetic data. <br>


We provide a pretrained model (*RealESRGAN_x4plus.pth*) with upsampling X4.<br>
**Note that RealESRGAN may still fail in some cases as the real-world degradations are really too complex.**<br>
Moreover, it **may not** perform well on **human faces, text**, *etc*, which will be optimized later.
<br>

You can also find a **Portable Windows/Linux/MacOS executable files for Intel/AMD/Nvidia GPU.** in our [GitHub repo](https://github.com/xinntao/Real-ESRGAN). <br>
This executable file is **portable** and includes all the binaries and models required. No CUDA or PyTorch environment is needed.<br>
This executable file is based on the wonderful [Tencent/ncnn](https://github.com/Tencent/ncnn) and [realsr-ncnn-vulkan](https://github.com/nihui/realsr-ncnn-vulkan) by [nihui](https://github.com/nihui).

# 1. Preparations
Before start, make sure that you choose
* Runtime Type = Python 3
* Hardware Accelerator = GPU

in the **Runtime** menu -> **Change runtime type**

Then, we clone the repository, set up the envrironment, and download the pre-trained model.

In [ ]:
# Clone Real-ESRGAN and enter the Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr
!pip install facexlib
!pip install gfpgan
!pip install -r requirements.txt
!python setup.py develop
# Download the pre-trained model
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P experiments/pretrained_models

# 2. Upload Images

Upload the images to be processed by Real-ESRGAN

In [ ]:
import os
from google.colab import files
import shutil

upload_folder = 'upload'
result_folder = 'results'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)
os.mkdir(upload_folder)
os.mkdir(result_folder)

# upload images
uploaded = files.upload()
for filename in uploaded.keys():
  dst_path = os.path.join(upload_folder, filename)
  print(f'move {filename} to {dst_path}')
  shutil.move(filename, dst_path)

In [ ]:
face = False

# 3. Inference x2 (for 512px to 1024px ~HD)



In [ ]:
# if it is out of memory, try to use the `--tile` option
# We upsample the image with the scale factor X3.5
import torch
torch.cuda.empty_cache()
if face:
  !python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 2 --face_enhance
else:
  !python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 2
# Arguments
# -n, --model_name: Model names
# -i, --input: input folder or image
# --outscale: Output scale, can be arbitrary scale factore.

# 3.1 Inference x8 (for 512px to 4096px ~4K)


In [ ]:
import torch
torch.cuda.empty_cache()
if face:
  !python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 4 --face_enhance --tile 1000
  !python inference_realesrgan.py -n RealESRGAN_x4plus -i results --outscale 2 --face_enhance --tile 1000
else:
  !python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 4 --tile 1000
  !python inference_realesrgan.py -n RealESRGAN_x4plus -i results --outscale 2 --tile 1000


# 4. Visualization

In [ ]:
import cv2
import matplotlib.pyplot as plt
def display(img1, img2):
  fig = plt.figure(figsize=(25, 10))
  ax1 = fig.add_subplot(1, 2, 1)
  plt.title('Input image', fontsize=16)
  ax1.axis('off')
  ax2 = fig.add_subplot(1, 2, 2)
  plt.title('Real-ESRGAN output', fontsize=16)
  ax2.axis('off')
  ax1.imshow(img1)
  ax2.imshow(img2)
def imread(img_path):
  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  return img

# display each image in the upload folder
import os
import glob

input_folder = 'upload'
result_folder = 'results'
input_list = sorted(glob.glob(os.path.join(input_folder, '*')))
output_list = sorted(glob.glob(os.path.join(result_folder, '*')))
for input_path, output_path in zip(input_list, output_list):
  img_input = imread(input_path)
  img_output = imread(output_path)
  display(img_input, img_output)

In [1]:
!touch /content/test_file.txt

In [2]:
!python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 4 --tile 512 -o /content/

python3: can't open file '/content/inference_realesrgan.py': [Errno 2] No such file or directory


In [3]:
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

sed: can't read /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py: No such file or directory


In [5]:
!find /usr/local/lib -name "degradations.py"

In [6]:
import basicsr
print(basicsr.__file__)
!ls -l /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py


ModuleNotFoundError: No module named 'basicsr'

In [17]:
from PIL import Image
import os

# Указываем путь к папке с результатами
folder_path = "/content/Real-ESRGAN/results/"

# Проверяем, существует ли папка
if os.path.exists(folder_path):
    # Перебираем все файлы в папке
    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(('.jpg', '.jpeg')):
            file_path = os.path.join(folder_path, file_name)

            # Открываем изображение и сохраняем как PNG
            with Image.open(file_path) as img:
                new_path = os.path.splitext(file_path)[0] + ".png"
                img.save(new_path, "PNG")
                print(f"Конвертирован: {file_name}")
    print("Готово! Все файлы обработаны.")
else:
    print("Папка не найдена. Проверь путь!")

Конвертирован: bby_light_out.jpeg
Конвертирован: bby_zlo_out.jpeg
Конвертирован: bby_funneral_out.jpeg
Конвертирован: bby_evil_out.jpeg
Конвертирован: bby_profile_out.jpeg
Готово! Все файлы обработаны.


In [7]:
# Удаляем старую папку, если она есть
!rm -rf Real-ESRGAN

# Клонируем заново
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Устанавливаем зависимости с принудительным обновлением
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install basicsr facexlib gfpgan
!python setup.py develop

Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 12.79 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/Real-ESRGAN
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 24.8 MB/s  0:00:00
  Created wheel for basicsr: filename=basicsr-1.4.2-py3-none-any.whl size=214907 sha256=682f0b7c73027d0207d445289e37fdc684a

In [14]:
!python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 4 --tile 512

Testing 0 bby_evil
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 1 bby_funneral
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 2 bby_house
	Tile 1/9
	Tile 2/9
	Tile 3/9
	Tile 4/9
	Tile 5/9
	Tile 6/9
	Tile 7/9
	Tile 8/9
	Tile 9/9
Testing 3 bby_light
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 4 bby_profile
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 5 bby_zlo
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4


In [11]:
!cp /content/Real-ESRGAN/results/* /content/


cp: cannot stat '/content/Real-ESRGAN/results/*': No such file or directory


In [12]:
!ls -R

.:
assets			       inputs		    requirements.txt
CODE_OF_CONDUCT.md	       LICENSE		    results
cog_predict.py		       MANIFEST.in	    scripts
cog.yaml		       options		    setup.cfg
docs			       README_CN.md	    setup.py
experiments		       README.md	    tests
inference_realesrgan.py        realesrgan	    VERSION
inference_realesrgan_video.py  realesrgan.egg-info  weights

./assets:
realesrgan_logo_ai.png	realesrgan_logo_gv.png	teaser-text.png
realesrgan_logo_av.png	realesrgan_logo.png
realesrgan_logo_gi.png	teaser.jpg

./docs:
anime_comparisons_CN.md  CONTRIBUTING.md  ncnn_conversion.md
anime_comparisons.md	 FAQ.md		  Training_CN.md
anime_model.md		 feedback.md	  Training.md
anime_video_model.md	 model_zoo.md

./experiments:
pretrained_models

./experiments/pretrained_models:
README.md

./inputs:
00003.png	0030.jpg	      OST_009.png	    wolf_gray.jpg
00017_gray.png	ADE_val_00000114.jpg  tree_alpha_16bit.png
0014.jpg	children-alpha.png    video

./inputs/video:
onepiece_demo.mp4

./op

In [13]:
!ls upload


ls: cannot access 'upload': No such file or directory


In [9]:
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

In [19]:
!python inference_realesrgan.py -n realesr-general-x4v3 -i upload -o results --outscale 4 --tile 512

Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-wdn-x4v3.pth" to /content/Real-ESRGAN/weights/realesr-general-wdn-x4v3.pth

100% 4.66M/4.66M [00:00<00:00, 94.9MB/s]
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth" to /content/Real-ESRGAN/weights/realesr-general-x4v3.pth

100% 4.66M/4.66M [00:00<00:00, 91.6MB/s]
Testing 0 bby_evil
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 1 bby_funneral
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 2 bby_house
	Tile 1/9
	Tile 2/9
	Tile 3/9
	Tile 4/9
	Tile 5/9
	Tile 6/9
	Tile 7/9
	Tile 8/9
	Tile 9/9
Testing 3 bby_light
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 4 bby_profile
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4
Testing 5 bby_zlo
	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4


In [25]:
import os
from PIL import Image
from rembg import remove

# 1. Настройки путей
input_dir = '/content/upload'
upscaled_dir = '/content/Real-ESRGAN/results_upscaled'
final_dir = '/content/Real-ESRGAN/results_final'

os.makedirs(upscaled_dir, exist_ok=True)
os.makedirs(final_dir, exist_ok=True)

# 2. Апскейл (используем модель, которая лучше сохраняет графические линии)
print("Начинаю апскейл...")
!python inference_realesrgan.py -n realesr-general-x4v3 -i {input_dir} -o {upscaled_dir} --outscale 4 --tile 512

# 3. Удаление фона и проверка пропорций
print("Обработка прозрачности и проверка вертикальной ориентации...")

for file_name in os.listdir(upscaled_dir):
    if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        file_path = os.path.join(upscaled_dir, file_name)

        with Image.open(file_path) as img:
            # Удаление фона для чистого принта
            img_rgba = img.convert("RGBA")
            subject = remove(img_rgba)

            # Проверка: вертикальный ли арт?
            width, height = subject.size
            if height < width:
                print(f"Внимание: {file_name} имеет горизонтальную ориентацию ({width}x{height})!")
            else:
                print(f"Файл {file_name} успешно обработан ({width}x{height})")

            # Сохранение итогового файла
            output_path = os.path.join(final_dir, os.path.splitext(file_name)[0] + '.png')
            subject.save(output_path, "PNG")

print(f"Готово! Чистые PNG для печати лежат в {final_dir}")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



No onnxruntime backend found.
Please install rembg with CPU or GPU support:

    pip install "rembg[cpu]"  # for CPU
    pip install "rembg[gpu]"  # for NVIDIA/CUDA GPU

For more information, see: https://github.com/danielgatis/rembg#installation
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/rembg/bg.py", line 9, in <module>
    import onnxruntime as ort  # type: ignore[import-untyped]
    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime/__init__.py", line 78, in <module>
    raise import_capi_exception
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime/__init__.py", line 26, in <module>
    from onnxruntime.capi._pybind_state import (
  File "/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/_pybind_state.py", line 32, in <module>
    from .onnxruntime_pybind11_state import *  # noqa
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: libcudart.so.13: cannot open shared object file: No suc

TypeError: object of type 'NoneType' has no len()

In [22]:
!pip install rembg

  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 137.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 99.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 52.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 109.5 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvml

In [24]:
!pip install "rembg[gpu]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 67.7 MB/s  0:00:03


In [26]:
!pip uninstall -y rembg onnxruntime onnxruntime-gpu
!pip install "rembg[cpu]"

Found existing installation: rembg 2.0.77
Uninstalling rembg-2.0.77:
  Successfully uninstalled rembg-2.0.77
Found existing installation: onnxruntime-gpu 1.27.0
Uninstalling onnxruntime-gpu-1.27.0:
  Successfully uninstalled onnxruntime-gpu-1.27.0
  Using cached rembg-2.0.77-py3-none-any.whl.metadata (18 kB)
Using cached rembg-2.0.77-py3-none-any.whl (46 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 39.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rembg]
